In [ ]:
# Cai dat dependency cho pipeline module hoa
%pip install -r requirements.txt -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.6/306.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 16.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.17.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.20.3, but you have protobuf 5.29.3 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.8/50.8 kB 2.9 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import os

from dotenv import load_dotenv

from pipeline.config import PipelineConfig
from pipeline.pipeline import build_vector_index, run_smoke_search

# Build Vector Database (Module-based Pipeline)
Notebook nay da duoc cap nhat de goi module `pipeline` thay vi viet inline tung buoc.
Flow: load config -> chunk -> embed -> upsert Qdrant -> smoke retrieval.

In [ ]:
# Dam bao dang dung thu muc Data
os.chdir(Path.cwd())
load_dotenv()

config = PipelineConfig.from_env()
config.source_dir = "./Database"

print("Current working dir:", Path.cwd())
print("Source dir:", config.source_dir)
print("Collection:", config.collection_name)
print("Chunk strategy:", config.chunk_strategy)

In [ ]:
# Build va index toan bo du lieu len Qdrant
result = build_vector_index(config)
vector_store = result["vector_store"]

print("Loaded documents:", len(result["raw_documents"]))
print("Chunked documents:", len(result["chunked_documents"]))
print("Indexed collection:", config.collection_name)

Downloading...
From: https://drive.google.com/uc?id=12U_2V2y5bUB9JLx9DdqDFmGH_OQQLW3o
To: /content/Data.zip
100%|██████████| 163k/163k [00:00<00:00, 67.2MB/s]


'Data.zip'

In [ ]:
# Smoke test retrieval
query = "Các môn học Học kỳ 2 năm 3 ngành hệ thống thông tin"
top_k = 5
found_docs = run_smoke_search(vector_store, query=query, top_k=top_k)

for i, doc in enumerate(found_docs, start=1):
    preview = doc.page_content[:200].replace("\n", " ")
    print(f"{i}. source={doc.metadata.get('source', 'unknown')} | {preview}")

## Tuy chinh cau hinh (optional)
Ban co the sua `.env` hoac override truc tiep trong cell ben duoi.

In [ ]:
# Optional override
config.collection_name = os.getenv("QDRANT_COLLECTION_NAME", config.collection_name)
config.chunk_strategy = os.getenv("CHUNK_STRATEGY", config.chunk_strategy)
config.chunk_size = int(os.getenv("CHUNK_SIZE", str(config.chunk_size)))
config.chunk_overlap = int(os.getenv("CHUNK_OVERLAP", str(config.chunk_overlap)))

# Validate key truoc khi index
config.validate()
print("Config ready.")